# Epic 9.03 — Full Pipeline: Generate, Evaluate, CLI

This notebook demonstrates the end-to-end workflow:
1. Generate a synthetic crack dataset
2. Load and inspect the dataset
3. Evaluate with crack metrics
4. CLI command reference

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic9.data.crack_dataset import CrackDataset, generate_crack_dataset
from udm_epic9.evaluation.crack_metrics import crack_detection_rate, crack_length_error

## 1. On-the-fly Dataset

The `CrackDataset` can generate samples on-the-fly without writing to disk.

In [ ]:
ds = CrackDataset(
    n_samples=20,
    image_size=(128, 128),
    domain='usm',
    seed=42,
    empty_fraction=0.1,
)

print(f'Dataset length: {len(ds)}')
sample = ds[0]
print(f'Image shape: {sample["image"].shape}')
print(f'Mask shape:  {sample["mask"].shape}')
print(f'Crack type:  {sample["crack_type"]}')
print(f'Domain:      {sample["domain"]}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    sample = ds[i]
    axes[0, i].imshow(sample['image'][0], cmap='gray')
    axes[0, i].set_title(sample['crack_type'], fontsize=9)
    axes[0, i].axis('off')
    axes[1, i].imshow(sample['mask'][0], cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Image')
axes[1, 0].set_ylabel('Mask')
plt.suptitle('On-the-fly CrackDataset Samples')
plt.tight_layout()
plt.show()

## 2. Crack Metrics

Evaluate crack detection rate and length error.

In [ ]:
# Simulate perfect predictions
true_masks = []
for i in range(10):
    sample = ds[i]
    true_masks.append(sample['mask'][0].numpy())

# Perfect detection
rate_perfect = crack_detection_rate(true_masks, true_masks)
print(f'Perfect detection rate: {rate_perfect:.2f}')

# Zero detection (all-zero predictions)
zero_masks = [np.zeros_like(m) for m in true_masks]
rate_zero = crack_detection_rate(zero_masks, true_masks)
print(f'Zero detection rate:    {rate_zero:.2f}')

# Length error for identical masks
if true_masks[0].sum() > 0:
    err = crack_length_error(true_masks[0], true_masks[0])
    print(f'Length error (identical): {err:.4f}')

## 3. Generate Dataset to Disk

Use `generate_crack_dataset` to write images, masks, and a manifest.

In [ ]:
# Generate a small dataset for demonstration
manifest_path = generate_crack_dataset(
    output_dir='/tmp/epic9_demo',
    n_samples=20,
    domains=['usm'],
    seed=42,
    image_size=(128, 128),
)
print(f'Manifest: {manifest_path}')

import json
with open(manifest_path) as f:
    manifest = json.load(f)
print(f'Samples: {manifest["n_samples"]}')
print(f'Splits: {manifest["splits"]}')

## 4. CLI Command Reference

```bash
# Generate full dataset
udm-epic9 generate --config configs/epic9_crack.yaml

# Generate from mask
udm-epic9 from-mask --mask crack.png --domain rgb --output out.png

# Domain transfer directory
udm-epic9 transfer --input-dir data/usm/ --output-dir data/rgb/

# Preview samples
udm-epic9 preview --n 8 --domain both

# Dataset statistics
udm-epic9 stats --config configs/epic9_crack.yaml

# Evaluate model
udm-epic9 evaluate --checkpoint model.pth --config configs/epic9_crack.yaml
```